# Example likelihood calculations for cluster probes (NC, CWL, Cxi2)

In [ ]:
# General imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy import integrate
from copy import deepcopy
import os,sys

from cloelib.cosmology.camb_cosmology import CAMBBackground, CAMBLinearPerturbations
from cloelib.observables.clusters.covariance import HaloCovariance
from cloelib.observables.clusters.halo_abundance import CastroHaloAbundance
from cloelib.observables.clusters.halo_clustering import TwoPoint3DHaloClustering
from cloelib.observables.clusters.halo_profile import NFWHaloProfile
from cloelib.observables.clusters.matter_statistics import MatterStatistics
from cloelib.observables.clusters.selection_function import GaussianSelectionFunction
from cloelib.observables.clusters.halo_mass_observable import (
    LognormalPowerLawHaloMassObservable,
)
from cloelib.summary_statistics.clusters import (
    ClusterClustering,
    ClusterCounts,
    ClusterStatisticsModeling,
    ClusterWeakLensing,
)

from cloelike.EuclidLikelihood_CG import EuclidLikelihood_CG

In [ ]:
import time

## Download Synthetic Data

We are currently developing homogeneous synthetic data vectors, mixing matrices, and covariance matrices that are compatible with [euclidlib](https://github.com/euclidlib/euclidlib) and adhere to the data format specifications used by the Euclid Science Ground Segment (LE3).

> ⚠️ This effort is still ongoing. Interfaces and file availability may change as development progresses.


In [ ]:
# # Execute this cell to download the data for testing the likelihood calculations
# data_downloaded = False

# if data_downloaded == False:
    
#     import requests
    
#     # URLs of the files to be downloaded
#     urls = {
#         'nz_example.fits': 'https://zenodo.org/records/15092862/files/nz_example.fits',
#         'cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy': 'https://zenodo.org/records/15496892/files/cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy',
#         'mixmat_identity_5000_binned.fits': 'https://zenodo.org/records/15496892/files/mixmat_identity_5000_binned.fits',
#         'synth_cells_5000_binned.fits': 'https://zenodo.org/records/15496892/files/synth_cells_5000_binned.fits',
#     }
    
#     # Function to download a file
#     def download_file(url, filename):
#         response = requests.get(url)
#         if response.status_code == 200:
#             with open(filename, 'wb') as f:
#                 f.write(response.content)
#             print(f'{filename} downloaded successfully')
#         else:
#             print(f'Failed to download {filename}. Status code: {response.status_code}')
    
#     # Download all files
#     for filename, url in urls.items():
#         download_file(url, filename)

## Read the Data and covariance matrices

<!-- To begin working with the photometric observables, the first step involves reading the data and normalising the redshift distribution, $n(z)$.

> **Note**: You will need to have installed `euclidlib` version **v2025.2**. Version **v2025.1** is also compatible.

Ensure you have the necessary input files (e.g., redshift distributions, masks, and bias functions) and follow the loading routines provided in the `euclidlib.photo` module. -->


In [ ]:
CG_CC = np.load('../data/clusters/data_CG_CC.npy')
CG_cov_CC = np.load('../data/clusters/data_cov_CG_CC.npy')
CG_MoR = np.load('../data/clusters/data_CG_MoR.npy')
CG_cov_MoR = np.load('../data/clusters/data_cov_CG_MoR.npy')
CG_xi2 = np.load('../data/clusters/data_CG_xi2.npy')
CG_cov_xi2 = np.load('../data/clusters/data_cov_CG_xi2.npy')

## Prepare Data Dictionary and settings

We envision that the preparation of data will become significantly simpler once a unified format for reading inputs is adopted. This standardised structure will allow for easier integration of synthetic data vectors, mixing matrices, and covariance matrices into the likelihood evaluation pipeline. 


In [ ]:
def build_data():
    
    data = {
        'CG_CC': CG_CC,
        'CG_cov_CC': CG_cov_CC,
        'CG_MoR': CG_MoR,
        'CG_cov_MoR': CG_cov_MoR,
        'CG_xi2': CG_xi2,
        'CG_cov_xi2': CG_cov_xi2,        
    }
    
    return data

def build_settings():
    
    settings = {            
            'overdensity_type': 'vir',
            'overdensity': 200.0,
            'CG_like_selection': 'CC_CWL_Cxi2',
            'CG_xi2_cov_selection': 'covCC_covCxi2',
            'bias': 'castro23',
            
            'r_interp': np.logspace(-10, 2.5, 200),
            'two_halo': 'None',
            'offcentering': False,
            'rms_off': 0.0,
            'f_off': 0.0,
            'trunc_fact': 2.5,
            'zs_max': 4.0,
            'mean_nz': 0.4,
            'sigma_nz': 0.3,
            'alpha_nz': 0.4,
            

            'zed_obs_edges': np.linspace(0.2, 1.8, 9),
            'Lambda_obs_edges': np.array([20.0, 30.0, 45.0, 60.0, 500.0]),
            'Rad_obs_edges': np.linspace(5.0, 100.0, 11),
            'Lambda_obs_Cxi2_edges': np.array([20, 30, 500]),
            'Rad_obs_Cxi2_edges': np.geomspace(20.0, 130.0, 31),
            'zed_obs_Cxi2_edges': np.arange(0.2, 1.81, 0.4),
            
            'k': np.geomspace(1e-4, 10, 500),
            'halo_concentration': 0.1,
            'Mass': np.logspace(12.0, 16.0, 51),
            'Lambda': np.geomspace(5.0, 500.0, 101),
            'zed': np.linspace(1.e-5, 4.0, 101),
            'area': 10313        
    }    
    return settings
    
data = build_data()
settings = build_settings()

## Likelihood Evaluation

In this section, we perform a single evaluation of the log-likelihood for the clusters probe at the fiducial cosmological parameters.

In [ ]:
# we call first a fiducial configuration that could serve us when we want to correct measurments taken at certain fiducial values
parameters_fid = {
'H0' :  67.32,
'omch2' : 0.120088016,
'Omega_cdm0' : 0.120088016/(67.32/100.0)**2,
'ombh2' : 0.022387993,
'Omega_b0' : 0.022387993/(67.32/100.0)**2,
'Omega_k0' : 0.0,
'w0' : -1.0,
'wa' : 0.0,
'ns' : 0.9661,
'mnu' : (0.3158-0.314378999)*93.04*pow(0.6732, 2),
'As' : 2.09175e-9,
'gamma_MG' : 0.545,
'N_mnu' : 1,
#'N_ur':2.046,   
}

start = time.time()
background_fid = CAMBBackground(
            H0=parameters_fid['H0'], Omega_cdm0=parameters_fid['Omega_cdm0'],
            Omega_b0=parameters_fid['Omega_b0'], Omega_k0=parameters_fid['Omega_k0'],
            w0=parameters_fid['w0'], wa=parameters_fid['wa'],  ns=parameters_fid['ns'],
            As=parameters_fid['As'], mnu=parameters_fid['mnu'],
            gamma_MG=parameters_fid['gamma_MG'], N_mnu=parameters_fid['N_mnu'])

end = time.time()
print(f"⏱️ Time elapsed: {end - start:.3f} seconds")

In [ ]:
parameters = {
'H0' :  67.32,
'omch2' : 0.118088016,
'Omega_cdm0' : 0.118088016/(67.32/100.0)**2,
'ombh2' : 0.022387993,
'Omega_b0' : 0.022387993/(67.32/100.0)**2,
'Omega_k0' : 0.0,
'w0' : -1.0,
'wa' : 0.0,
'ns' : 0.9661,
'mnu' : 0.05991730896,
'As' : 2.09175e-9,
'gamma_MG' : 0.545,
'N_mnu' : 1,
}

_hmo_pars = {
'A_l': 52.0,
'B_l': 0.9,
'C_l': 0.5,
'sig_A_l': 0.2,
'sig_B_l': -0.05,
'sig_C_l': 0.001 ,
}

_sel_pars = {
'sig_lambda_norm': 0.9,
'sig_lambda_z': 0.1 ,
'sig_lambda_exponent': 0.4,
'sig_z_z': 0.025,
'sig_z_lambda': 5.e-6
}

cluster_lkl = EuclidLikelihood_CG(data, settings, _hmo_pars, _sel_pars, CAMBBackground, CAMBLinearPerturbations, LognormalPowerLawHaloMassObservable,
                                  GaussianSelectionFunction, MatterStatistics, CastroHaloAbundance, NFWHaloProfile, TwoPoint3DHaloClustering,
                                  HaloCovariance, ClusterStatisticsModeling, ClusterCounts, ClusterWeakLensing, ClusterClustering, background_fid)
start = time.time()
print(cluster_lkl.loglike(parameters, _hmo_pars, _sel_pars, settings))
end = time.time()
print(f"⏱️ Time elapsed: {end - start:.3f} seconds")

## Exploration of the Parameter Space

This section illustrates how we explore the dependence of clusters observables on the cold dark matter density parameter, $\Omega_{\rm cdm} h^2$. We scan a range of values and evaluate the likelihood for each the clusters probe

In [ ]:
# Define the range of Omega_cdm0 values to scan
# omch2_vals = np.linspace(0.1160, 0.1211, 100)

# likelihood_curve = []

# for omch2 in omch2_vals:
#         # Update Omega_cdm0 parameter from omch2 and H0
#         parameters['Omega_cdm0'] = omch2 / (parameters['H0'] / 100.0)**2
#         start = time.time()
#         # Compute log-likelihood
#         ll = cluster_lkl.loglike(parameters, _hmo_pars, _sel_pars, settings)
#         end = time.time()
#         print(f"⏱️ Time elapsed: {end - start:.3f} seconds")
#         likelihood_curve.append(ll)

## Plot

Here we normalize each resulting curve to its maximum value for easier visual comparison. A vertical dashed line indicates a fiducial $\Omega_{\rm cdm} h^2$ value (e.g., the best fit).

In [ ]:
# # Convert log-likelihoods to likelihoods (exp) and normalize
# likelihood_curve = np.exp(likelihood_curve - np.max(likelihood_curve))

# # Prepare the figure
# plt.figure(figsize=(8,6))
# label = settings['CG_like_selection']

# # Plot
# plt.plot(omch2_vals, likelihood_curve, label=label)

# # Reference vertical line 
# plt.axvline(0.120088016, color='k', linestyle='--', label='Reference $\Omega_{cdm} h^2$')

# plt.xlabel(r'$\Omega_{cdm} h^2$')
# plt.ylabel('Normalized Likelihood')
# plt.title('Likelihood scan over $\Omega_{cdm} h^2$ for photometric probes')
# plt.legend()
# plt.xlim([0.1201-0.0005, 0.1201+0.0004])
# plt.grid(True)
# plt.tight_layout()
# plt.show()